# Ensure ZeroBus Service Principal

Creates (or finds) a least-privilege service principal dedicated to the ZeroBus ingestion pipeline. The SPN is scoped to a single schema and secret scope — it cannot be reused for unrelated workloads.

This notebook runs as the **first task** in the `wearables_uc_setup` job. It outputs the SPN's `application_id` as a **task value** so the downstream DDL task can grant it table-level permissions.

**Secret scope keys auto-provisioned by this notebook:**

| Key | Source | Purpose |
| --- | --- | --- |
| `client_id` | SPN `application_id` | OAuth M2M client identifier |
| `workspace_url` | Derived from config | Databricks workspace URL for SDK/API calls |
| `zerobus_endpoint` | Derived from workspace ID + region | ZeroBus Ingest server endpoint |
| `target_table_name` | From job params | Fully qualified bronze table name |

**Secret scope key requiring manual admin provisioning:**

| Key | Source | Purpose |
| --- | --- | --- |
| `client_secret` | Admin-generated | OAuth M2M client secret |

> **Admin step required:** After the first run of this job, an admin must generate an OAuth secret for the SPN and populate the `client_secret` key in the secret scope. This can be done via:
> * **Workspace UI:** Settings → Identity and access → Service principals → select the SPN → Secrets → Generate secret
> * **Account Console:** User management → Service principals → Credentials & secrets
> * **Databricks CLI:** `databricks account service-principal-secrets create <workspace_sp_id>`
> * **External keystore:** Sync credentials from Azure Key Vault, AWS Secrets Manager, or HashiCorp Vault
>
> The `client_id` is stored automatically. Only the `client_secret` needs manual provisioning.

**Permissions handled here:**
* Secret scope `READ` ACL — so the Databricks App can retrieve all secrets at runtime

**Permissions handled by the DDL task (downstream):**
* `USE CATALOG`, `USE SCHEMA`, `MODIFY`, `SELECT` on the target table

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")

print(f"Catalog:      {catalog_use}")
print(f"Schema:       {schema_use}")
print(f"Secret scope: {secret_scope_name}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Naming convention: dbxw-zerobus-{schema} — ties the SPN to this schema
spn_display_name = f"dbxw-zerobus-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

# Search for an existing SPN with this name
existing_spns = list(
    w.service_principals.list(filter=f'displayName eq "{spn_display_name}"')
)

if existing_spns:
    spn = existing_spns[0]
    is_new_spn = False
    print(f"Found existing SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")
    print(f"  active: {spn.active}")
else:
    spn = w.service_principals.create(
        display_name=spn_display_name,
        active=True
    )
    is_new_spn = True
    print(f"Created new SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")

spn_application_id = spn.application_id

In [0]:
# ---------------------------------------------------------------------------
# Store derived values and the SPN's client_id in the secret scope, then
# verify that the admin-provisioned client_secret is present.
#
# Auto-provisioned: client_id, workspace_url, zerobus_endpoint, target_table_name
# These are refreshed on every run.
#
# Admin-provisioned: client_secret only
# Must be created via workspace UI, account console, CLI, or external keystore.
# ---------------------------------------------------------------------------

# ---- Derive ZeroBus endpoint from workspace metadata ---------------------
# Format: https://<workspace_id>.zerobus.<region>.cloud.databricks.com
# Ref: https://docs.databricks.com/aws/en/ingestion/zerobus-ingest/

workspace_url = w.config.host.rstrip("/")
workspace_id = None

# Try multiple methods to get the workspace ID
try:
    workspace_id = w.get_workspace_id()
except Exception:
    pass

if not workspace_id:
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        workspace_id = ctx.workspaceId().get()
    except Exception:
        pass

if not workspace_id:
    try:
        workspace_id = spark.conf.get("spark.databricks.workspaceUrl").split(".")[0]
    except Exception:
        pass

# Determine the AWS region for the ZeroBus endpoint
try:
    region = spark.conf.get("spark.databricks.clusterUsageTags.region", "")
except Exception:
    region = ""

if not region:
    # Fallback: extract from workspace URL hostname
    # Pattern: <name>.<region>.cloud.databricks.com
    host = workspace_url.replace("https://", "")
    parts = host.split(".")
    if len(parts) == 5 and parts[1] not in ("cloud",):
        region = parts[1]
    else:
        region = "us-east-1"  # AWS default region

zerobus_endpoint = f"https://{workspace_id}.zerobus.{region}.cloud.databricks.com"
target_table = f"{catalog_use}.{schema_use}.wearables_zerobus"

print(f"Workspace URL:     {workspace_url}")
print(f"Workspace ID:      {workspace_id}")
print(f"Region:            {region}")
print(f"ZeroBus endpoint:  {zerobus_endpoint}")
print(f"Target table:      {target_table}")

# ---- Store auto-provisioned values in the secret scope --------------------
try:
    secrets_to_store = {
        "client_id":        spn_application_id,
        "workspace_url":    workspace_url,
        "zerobus_endpoint": zerobus_endpoint,
        "target_table_name": target_table,
    }
    for key, value in secrets_to_store.items():
        w.secrets.put_secret(scope=secret_scope_name, key=key, string_value=value)
    print(f"\nStored in scope '{secret_scope_name}':")
    print(f"  client_id         = {spn_application_id}")
    print(f"  workspace_url     = {workspace_url}")
    print(f"  zerobus_endpoint  = {zerobus_endpoint}")
    print(f"  target_table_name = {target_table}")
except Exception as e:
    print(f"\nWarning: could not store secrets (scope may not exist yet): {e}")

# ---- Check for admin-provisioned client_secret ----------------------------
try:
    existing_keys = {s.key for s in w.secrets.list_secrets(scope=secret_scope_name)}
    credentials_provisioned = "client_secret" in existing_keys
    print(f"\nSecret scope keys: {sorted(existing_keys)}")
    if credentials_provisioned:
        print("OAuth client_secret: PRESENT")
    else:
        print("OAuth client_secret: MISSING")
        print()
        print("=" * 65)
        print("ACTION REQUIRED: An admin must provision the client_secret.")
        print("=" * 65)
        print()
        print(f"SPN display name:   {spn_display_name}")
        print(f"SPN application_id: {spn_application_id} (already stored as client_id)")
        print(f"SPN workspace ID:   {spn.id}")
        print()
        print("Option 1 \u2014 Workspace UI:")
        print("  Settings > Identity and access > Service principals")
        print(f"  > Select '{spn_display_name}' > Secrets > Generate secret")
        print()
        print("Option 2 \u2014 Databricks CLI (requires account admin):")
        print(f"  databricks account service-principal-secrets create {spn.id}")
        print()
        print("Then store the secret:")
        print(f'  databricks secrets put-secret --scope {secret_scope_name} --key client_secret --string-value "<secret>"')
except Exception:
    credentials_provisioned = False
    print(f"\nCould not check credentials (scope '{secret_scope_name}' may not exist yet).")

In [0]:
from databricks.sdk.service.workspace import AclPermission

# Grant the SPN READ access to the secret scope so the Databricks App
# can retrieve client_id, client_secret, and endpoint credentials at runtime.
# put_acl is idempotent — safe to re-run.
try:
    w.secrets.put_acl(
        scope=secret_scope_name,
        principal=spn_application_id,
        permission=AclPermission.READ
    )
    print(f"Granted READ on scope '{secret_scope_name}' to {spn_application_id}")
except Exception as e:
    # If the scope doesn't exist yet (first deploy before bundle creates it),
    # log the error but don't fail — the scope will be created by the bundle.
    print(f"Warning: Could not set secret scope ACL: {e}")
    print("The secret scope may not exist yet. Re-run after bundle deploy.")

In [0]:
# Pass the SPN application_id to the next task in the job.
# The DDL task reads this via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
print(f"Task value set: spn_application_id = {spn_application_id}")

In [0]:
print("=" * 65)
print("Service Principal Summary")
print("=" * 65)
print(f"  Display name:      {spn_display_name}")
print(f"  Application ID:    {spn_application_id}")
print(f"  Newly created:     {is_new_spn}")
print(f"  Secret scope:      {secret_scope_name}")
print(f"    READ ACL:        granted")
print(f"    client_id:       stored")
print(f"    workspace_url:   stored")
print(f"    zerobus_endpoint: stored")
print(f"    target_table:    stored")
print(f"    client_secret:   {'PRESENT' if credentials_provisioned else 'MISSING (admin action required)'}")
print(f"  ZeroBus endpoint:  {zerobus_endpoint}")
print(f"  Target table:      {target_table}")
print(f"  Target schema:     {catalog_use}.{schema_use}")
print()
if not credentials_provisioned:
    print("*** The client_secret must be provisioned by an admin  ***")
    print("*** before the Databricks App can authenticate.        ***")
    print()
print("UC grants (USE CATALOG, USE SCHEMA, MODIFY, SELECT)")
print("will be applied by the downstream DDL task.")
print("=" * 65)